# Practice run of analysing/testing different models on the UNSW_NB15 dataset, before trying Deep Learning.

Prior research suggests this is a largely non-linear, less separable dataset so deep learning may be necessary, but I will try simpler, more interpretable models first for the sake of completeness, and to gain Variable Importances

In [1]:
#import packages:


from google.colab import drive

try:
  import google.colab
  IN_COLAB = True
except:
  IN_COLAB = False

if IN_COLAB:
  # Check if drive is mounted by looking for the mount point in the file system.
  # This is a more robust approach than relying on potentially internal variables.
  import os
  if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
  else:
    print("Google Drive is already mounted.")
else:
  print("Not running in Google Colab. Drive mounting skipped.")

from IPython import get_ipython
from IPython.display import display
import cudf
import cuml
from cuml.preprocessing import StandardScaler
from cuml.model_selection import StratifiedKFold, GridSearchCV
from cuml.linear_model import LogisticRegression
from cuml.pipeline import Pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm


print("New run: Packages loaded")

Google Drive is already mounted.
New run: Packages loaded


Let's load our packages and data

In [2]:
#if using colabs - will need to first mount your drive

#change these for different users
test_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_testing-set.parquet'
training_set_filepath = '/content/drive/MyDrive/Colab_Notebooks/Data/UNSW_NB15_training-set.parquet'

# Import the two CSV files
test_set = pd.read_parquet(test_set_filepath)
train_set = pd.read_parquet(training_set_filepath)

print("Data loaded")


Data loaded


The next cell does some basic analysis, and one hot encodes some of the features:

In [3]:
def preprocess_data(data_set):
    # Remove 'attack_cat' column if it exists
    if 'attack_cat' in data_set.columns.tolist():
        data_set.drop('attack_cat', axis=1, inplace=True)

    if 'proto' in data_set.columns.tolist():
        # Ensure 'proto' is of type 'object' to avoid categorical issues
        data_set['proto'] = data_set['proto'].astype(str)

        # Calculate percentage occurrences of each category
        category_percentages = data_set['proto'].value_counts(normalize=True) * 100
        top_6_categories = category_percentages.head(6).index.tolist()

        # Group less frequent categories under 'other' using vectorized operations
        data_set['proto_grouped'] = data_set['proto']
        data_set.loc[~data_set['proto'].isin(top_6_categories), 'proto_grouped'] = 'other'

        # Drop the original 'proto' column
        data_set.drop('proto', axis=1, inplace=True)

        # One-hot encode the 'proto_grouped' column
        data_set = pd.get_dummies(data_set, columns=['proto_grouped'], prefix='proto_grouped')

    # One-hot encode any remaining categorical columns
    categorical_cols = data_set.select_dtypes(include=['object', 'category']).columns.tolist()
    if categorical_cols:
        data_set = pd.get_dummies(data_set, columns=categorical_cols, prefix_sep='_')

    # Convert boolean columns to integers
    binary_cols = data_set.select_dtypes(include=['bool']).columns

    if not binary_cols.empty:
        data_set[binary_cols] = data_set[binary_cols].astype(int)

    return data_set


train_set = preprocess_data(train_set)
test_set = preprocess_data(test_set)

print(f"Data set preprocessed, columns = {len(train_set.columns.tolist())}")
print(f"Data set preprocessed, columns = {len(test_set.columns.tolist())}")


#turn both pandas dataframes into cudf
train_set = cudf.DataFrame.from_pandas(train_set)
test_set = cudf.DataFrame.from_pandas(test_set)

print("Data preprocessed")

Data set preprocessed, columns = 61
Data set preprocessed, columns = 59
Data preprocessed


NOTE TO SELF -
1. THIS IS FOR BINARY CLASSIFICATION, WE WANT MULTICLASS EVENTUALLY, BUT FOR NOW WE WILL JUST DO BN


Based on the high number of columns in the Proto column, we may want to consider an Embeddings layer with the Deep Learning that we plan to undertake later. However since DT/RF perform somewhat poorly on sparse vector datasets (like one hot encoded ones) we will group all the extremely rare categories into an 'other'.


In [6]:
def run_models(model_type, test_set=test_set, train_set=train_set):
    """
    Runs LR, DT or RF model on dataframe
    """
    #drop label and define list of out targets
    X_train = train_set.drop('label', axis=1)
    y_train = train_set['label']

    #do the same for test set
    X_test = test_set.drop('label', axis=1)
    y_test = test_set['label']

    # List only the categorical columns (object types)
    categorical_cols = train_set.select_dtypes(include=['category']).columns.tolist()

    #=============================== LR ========================================#
    if model_type.upper() == 'LR':
        # Define the pipeline
        pipeline = Pipeline([
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(max_iter=1000))
        ])

        # Define the hyperparameter grid
        param_grid = {
            'classifier__C': [0.01, 0.1, 1, 10, 100],
            'classifier__penalty': ['l1', 'l2'],
            'classifier__solver': ['liblinear']  # Suitable for smaller datasets
        }

        # Set up cross-validation strategies using cuML's StratifiedKFold
        inner_cv = cuml.model_selection.StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
        outer_cv = cuml.model_selection.StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        print("KFolds defined, beginning nested cross-validation.")

        # Outer loop (on training data)
        outer_scores = []

        # outer loop
        for train_index, val_index in tqdm(outer_cv.split(X_train, y_train)):
            X_train_fold, X_val_fold = X_train.iloc[train_index], X_train.iloc[val_index]
            y_train_fold, y_val_fold = y_train.iloc[train_index], y_train.iloc[val_index]

            # Set up GridSearchCV with the pipeline (inner loop) - USE cuML VERSION
            print("Some important types:")
            print(type(Pipeline))
            print(type(StandardScaler()))
            print(type(LogisticRegression()))
            print(type(GridSearchCV))


            grid_search = cuml.model_selection.GridSearchCV(  # Use cuML's GridSearchCV
                estimator=pipeline,
                param_grid=param_grid,
                cv=inner_cv,
                scoring='roc_auc',  # Use appropriate scoring for your task
            )

            # Fit GridSearchCV on the training fold
            grid_search.fit(X_train_fold, y_train_fold)  # Now using cuDF data

            # Evaluate on the validation fold
            score = grid_search.score(X_val_fold, y_val_fold)  # Now using cuDF data
            outer_scores.append(score)

        # Print the average outer score (ROC AUC) on training data
        print(f"Average Validation ROC AUC: {np.mean(outer_scores)}")

        # Evaluate the best model on the held-out test set
        best_model = grid_search.best_estimator_
        test_score = best_model.score(X_test, y_test)  # Now using cuDF data
        print(f"Test ROC AUC: {test_score}")

    #=============================== DT ========================================#
    if model_type.upper() == 'DT':
        param_grid_dt = {
            'max_depth': [3, 5, 10],             # List of max depth values
            'min_samples_split': [2, 10, 20],    # List of minimum samples to split
            'min_samples_leaf': [1, 5, 10]       # List of minimum samples per leaf
        }
        pass  # Placeholder for DT logic

    #=============================== RF ========================================#
    if model_type.upper() == 'RF':
        param_grid_rf = {
            'n_estimators': [50, 100, 200],      # List of number of trees
            'max_depth': [3, 5, 10],             # List of max depth values
            'min_samples_split': [2, 10, 20],    # List of minimum samples to split
            'min_samples_leaf': [1, 5, 10]
        }

        pass  # Placeholder for RF logic

run_models('LR')

KFolds defined, beginning nested cross-validation.


0it [00:00, ?it/s]

Some important types:
<class 'abc.ABCMeta'>
<class 'cuml._thirdparty.sklearn.preprocessing._data.StandardScaler'>
<class 'cuml.linear_model.logistic_regression.LogisticRegression'>
<class 'abc.ABCMeta'>


TypeError: StratifiedKFold.get_n_splits() got an unexpected keyword argument 'groups'